# 自定义中间件-Wrap-style hooks
## 1、wrap_model_call的使用

### 1.1 基于装饰器的实现

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

In [2]:

from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse, AgentMiddleware


@wrap_model_call
def wrap_model_call_middleware(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse | None:
    request.messages[-1].content += "---> wrap_model_call_before <---"

    # 模型的调用
    response = handler(request)

    response.result[0].content += "---> wrap_model_call_after <---"

    return response

In [3]:

from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    middleware=[
        wrap_model_call_middleware,
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好---> wrap_model_call_before <---
================================== Ai Message ==================================

你好！你是想让我帮你处理 `wrap_model_call_before` 相关的内容吗？  
如果你是在测试标签/占位符，我也可以直接继续配合。---> wrap_model_call_after <---


### 1.2 基于类的实现

In [4]:
from langchain.agents.middleware import AgentMiddleware


class WrapModelCallMiddleware(AgentMiddleware):
    def wrap_model_call(self, request: ModelRequest,
                        handler: Callable[[ModelRequest], ModelResponse],
                        ) -> ModelResponse | None:
        request.messages[-1].content += "---> wrap_model_call_before <---"

        # 模型的调用
        response = handler(request)

        response.result[0].content += "---> wrap_model_call_after <---"

        return response


agent = create_agent(
    model=model,
    middleware=[
        WrapModelCallMiddleware(),
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好---> wrap_model_call_before <---
================================== Ai Message ==================================

你好！有什么我可以帮你的吗？---> wrap_model_call_after <---


## 2、wrap_tool_call的使用

### 2.1 基于装饰器的实现

In [5]:

from langchain_core.tools import tool
from typing import Any
from langgraph.types import Command
from langchain_core.messages import ToolMessage
from langgraph.prebuilt.tool_node import ToolCallRequest
from langchain.agents.middleware import wrap_tool_call


@tool
def get_weather(city: str, is_forcast: bool) -> str:
    """
    获取当日特定城市的天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明天的天气预报
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天天气也很好"
    return res


@wrap_tool_call
def wrap_tool_call_middleware(request: ToolCallRequest,
                              handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                              ) -> ToolMessage | Command[Any]:
    result = handler(request)
    print(f"原始参数：{request.tool_call['args']}")
    print(f"原始参数调用结果：{result}")

    request.tool_call["args"]["is_forcast"] = True
    result = handler(request)

    print(f"更新以后的参数：{request.tool_call['args']}")
    print(f"更新以后的参数调用结果：{result}")
    return result


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[wrap_tool_call_middleware]
)

response = agent.invoke({
    "messages": [HumanMessage("帮我查询北京今天的天气如何？")]
})

for msg in response["messages"]:
    msg.pretty_print()

原始参数：{'city': '北京', 'is_forcast': False}
原始参数调用结果：content='北京今天天气不错' name='get_weather' tool_call_id='call_MCqn74Cp1iSMbr7mdh9ILS2A'
更新以后的参数：{'city': '北京', 'is_forcast': True}
更新以后的参数调用结果：content='北京今天天气不错\n明天天气也很好' name='get_weather' tool_call_id='call_MCqn74Cp1iSMbr7mdh9ILS2A'
================================ Human Message =================================

帮我查询北京今天的天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_MCqn74Cp1iSMbr7mdh9ILS2A)
 Call ID: call_MCqn74Cp1iSMbr7mdh9ILS2A
  Args:
    city: 北京
    is_forcast: True
================================= Tool Message =================================
Name: get_weather

北京今天天气不错
明天天气也很好
================================== Ai Message ==================================

北京今天天气不错。  
如果你需要，我也可以继续帮你整理成更适合出行的建议，比如是否适合带伞、穿什么衣服。


### 2.2 基于类的实现

In [6]:

class WrapToolCallMiddleware(AgentMiddleware):
    def wrap_tool_call(self, request: ToolCallRequest,
                       handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                       ) -> ToolMessage | Command[Any]:
        result = handler(request)
        print(f"原始参数：{request.tool_call['args']}")
        print(f"原始参数调用结果：{result}")

        request.tool_call["args"]["is_forcast"] = True
        result = handler(request)

        print(f"更新以后的参数：{request.tool_call['args']}")
        print(f"更新以后的参数调用结果：{result}")
        return result


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[WrapToolCallMiddleware()]
)

response = agent.invoke({
    "messages": [HumanMessage("帮我查询上海今天的天气如何？")]
})

for msg in response["messages"]:
    msg.pretty_print()

原始参数：{'city': '上海', 'is_forcast': False}
原始参数调用结果：content='上海今天天气不错' name='get_weather' tool_call_id='call_m2ft9YtgbGaWqXs5q2J6aB7D'
更新以后的参数：{'city': '上海', 'is_forcast': True}
更新以后的参数调用结果：content='上海今天天气不错\n明天天气也很好' name='get_weather' tool_call_id='call_m2ft9YtgbGaWqXs5q2J6aB7D'
================================ Human Message =================================

帮我查询上海今天的天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_m2ft9YtgbGaWqXs5q2J6aB7D)
 Call ID: call_m2ft9YtgbGaWqXs5q2J6aB7D
  Args:
    city: 上海
    is_forcast: True
================================= Tool Message =================================
Name: get_weather

上海今天天气不错
明天天气也很好
================================== Ai Message ==================================

上海今天天气不错，适合外出。  
如果你需要，我也可以继续帮你看一下明天的天气建议穿什么。
